# 01 — Ingest & Parse
This mirrors `examples/01_ingest_local_files.py`.


In [ ]:
from pathlib import Path
Path('data/samples').mkdir(parents=True, exist_ok=True)
Path('data/samples/acme_note.txt').write_text('ACME PLC reported a 38.2% gross margin in FY2024.', encoding='utf-8')
print('Seeded data/samples/acme_note.txt')

In [ ]:
import json
from dataclasses import dataclass, asdict
from pathlib import Path

def normalize_text(s: str) -> str:
    return ' '.join((s or '').replace('-\n','').replace('\n',' ').split())

@dataclass
class IngestResult:
    path: str; kind: str; ok: bool
    error: str|None=None; chars: int|None=None; rows: int|None=None; cols: int|None=None; sheets: int|None=None

def kind_from_suffix(path: str) -> str:
    s = Path(path).suffix.lower()
    return {'.pdf':'pdf','.docx':'docx','.txt':'txt','.csv':'csv','.tsv':'tsv','.xlsx':'xlsx','.json':'json','.jsonl':'jsonl'}.get(s,'unknown')

def ingest_path(p: Path) -> IngestResult:
    try:
        t = normalize_text(p.read_text(encoding='utf-8', errors='ignore'))
        return IngestResult(str(p), kind_from_suffix(str(p)), True, chars=len(t))
    except Exception as e:
        return IngestResult(str(p), kind_from_suffix(str(p)), False, error=str(e)[:400])

def scan(base: Path):
    pats = ['*.pdf','*.csv','*.tsv','*.xlsx','*.json','*.jsonl','*.txt','*.docx','*.md']
    out=set()
    for pat in pats: out.update(base.rglob(pat))
    return sorted(out)

base = Path('data/samples')
files = scan(base)
results = [ingest_path(p) for p in files]
Path('data/outputs').mkdir(parents=True, exist_ok=True)
out = Path('data/outputs/ingest_report.jsonl')
with out.open('w', encoding='utf-8') as f:
    for r in results: f.write(json.dumps(asdict(r), ensure_ascii=False)+'\n')
print('Wrote', out)